In [0]:
import google.generativeai as genai
from pyspark.sql.functions import col, desc, count, sum, when
import time
import os
from dotenv import load_dotenv

# --- AUTENTICAÇÃO ---
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    from getpass import getpass
    API_KEY = getpass("Cole sua GEMINI_API_KEY aqui: ")

genai.configure(api_key=API_KEY)

# --- SELETOR DE MODELO ---
def escolher_melhor_modelo():
    print("Verificando disponibilidade de modelos na API")
    try:
        modelos_disponiveis = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
        
        
        preferencias = ['models/gemini-2.5-flash', 'models/gemini-2.5-pro', 'models/gemini-1.5-pro', 'models/gemini-pro']
        
        for modelo in preferencias:
            if modelo in modelos_disponiveis:
                print(f"Modelo escolhido: {modelo}")
                return genai.GenerativeModel(modelo)
        
        if modelos_disponiveis:
            print(f"Usando fallback: {modelos_disponiveis[0]}")
            return genai.GenerativeModel(modelos_disponiveis[0])
            
    except Exception as e:
        print(f"Erro ao listar modelos: {e}")
        return genai.GenerativeModel('gemini-2.5-flash')

    raise Exception("Nenhum modelo disponível encontrado para esta API Key.")


model = escolher_melhor_modelo()

# --- 3. PREPARAÇÃO DOS DADOS ---
print("\n🚀 Preparando dados Olist...")
df_raw = spark.read.table("olist_portfolio.gold.reviews_analytics")

# Calcula KPIs
df_kpis = df_raw \
    .filter(col("product_category_name").isNotNull()) \
    .groupBy("product_category_name") \
    .agg(
        count("*").alias("total_reviews"),
        sum(when(col("sentiment_label") == "negativo", 1).otherwise(0)).alias("qtd_negativos")
    ) \
    .withColumn("taxa_reprovacao", col("qtd_negativos") / col("total_reviews")) \
    .filter(col("total_reviews") > 30) \
    .filter(col("taxa_reprovacao") >= 0.40) \
    .orderBy(desc("taxa_reprovacao"))

categorias_criticas = [row['product_category_name'] for row in df_kpis.collect()]
diagnosticos = []

if not categorias_criticas:
    print("Nenhuma categoria crítica hoje.")
else:
    print(f"Alvos: {categorias_criticas}")

# --- 4. LOOP DE INTELIGÊNCIA ---
for categoria in categorias_criticas:
    print(f"\n🔍 Analisando: {categoria.upper()}")
    
    # Coleta reviews
    raw_data = df_raw.filter(
        (col("product_category_name") == categoria) & 
        (col("sentiment_label") == "negativo")
    ).select("clean_text").limit(100).collect()
    
    lista_reviews = [r['clean_text'].strip() for r in raw_data if r['clean_text'] and len(r['clean_text']) > 15]
    texto_input = "\n- ".join(lista_reviews)
    
    if not texto_input:
        print("Sem dados suficientes.")
        continue

    # Prompt
    prompt = f"""
    Aja como Especialista em CX. Analise as reclamações da categoria '{categoria}'.
    1. Identifique os 3 principais problemas.
    2. Escreva um resumo executivo de 1 parágrafo (max 50 palavras) em Português Formal sobre a dor principal.
    3. Sem nomes de clientes.

    Reclamações:
    {texto_input[:25000]}
    """
    
    try:
        response = model.generate_content(prompt)
        analise = response.text
        
        taxa = df_kpis.filter(col("product_category_name") == categoria).first()['taxa_reprovacao']
        
        print(f"   🤖 Diagnóstico:\n{analise}")
        print("-" * 50)
        
        diagnosticos.append((categoria, float(taxa), analise))
        time.sleep(4) 
        
    except Exception as e:
        print(f"    Erro na API: {e}")

# --- 5. SALVAMENTO ---
if diagnosticos:
    print("\nSalvando tabela...")
    schema = "categoria STRING, taxa_reprovacao FLOAT, diagnostico_ia STRING"
    df_save = spark.createDataFrame(diagnosticos, schema)
    df_save.write.mode("overwrite").saveAsTable("olist_portfolio.gold.ai_diagnostics")
    print("Sucesso!")